# MiDA end-to-end pipeline (Colab-ready)

این نوت‌بوک تمام کدهای پروژه را در قالبی واحد ارائه می‌کند تا بتوانید آن را به‌راحتی در Google Colab اجرا کنید. سطرهای زیر همان منطق فایل‌های `MiDA.py`، `read_log.py`، `test.py` و `main.py` را پیاده‌سازی می‌کنند.

## ۱. نصب پیش‌نیازها

در محیط‌های ابری مثل Colab بهتر است کتابخانه‌های موردنیاز در یک سلول جداگانه نصب شوند. در صورت نصب بودن می‌توانید این سلول را رد کنید.

In [ ]:

# اگر از Colab استفاده می‌کنید این سلول را اجرا کنید.
!pip install --quiet numpy pandas scikit-learn tensorflow ConfigSpace smac


## ۲. وارد کردن کتابخانه‌ها و پیکربندی‌های عمومی

In [ ]:

import logging
from pathlib import Path
from time import perf_counter
from datetime import datetime
import pickle

import numpy as np
import pandas as pd
from pandas.api.types import is_numeric_dtype
from sklearn import preprocessing
from sklearn.metrics import classification_report, precision_recall_fscore_support
from sklearn.preprocessing import LabelBinarizer, MinMaxScaler
from sklearn.metrics import roc_auc_score, average_precision_score

from tensorflow.keras.layers import (BatchNormalization, Dense, Embedding, Input,
                                     LSTM, Reshape, concatenate)
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.optimizers import Nadam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

from ConfigSpace.hyperparameters import (CategoricalHyperparameter,
                                         UniformFloatHyperparameter)
from smac.configspace import ConfigurationSpace
from smac.scenario.scenario import Scenario
from smac.facade.hyperband_facade import HB4AC


## ۳. کلاس خواندن و آماده‌سازی لاگ (`ReadLog`)

این بخش معادل `read_log.py` است و مسئول آماده‌سازی دیدگاه‌های عددی و دسته‌ای می‌باشد.

In [ ]:

class ReadLog:
    def __init__(self, eventlog: str, data_root: Path | str = "fold"):
        self._eventlog = eventlog
        self._data_root = Path(data_root)
        self._data_dir = self._data_root / self._eventlog
        self._list_cat_cols: list[str] = []
        self._list_num_cols: list[str] = []
        self._data_dir.mkdir(parents=True, exist_ok=True)

    @staticmethod
    def union(lst1, lst2):
        return list(set(lst1) | set(lst2))

    @staticmethod
    def to_sec(delta):
        return (86400 * delta.days + delta.seconds + delta.microseconds / 1_000_000) / 86400

    @staticmethod
    def time_format(time_stamp: str):
        try:
            date_format_str = '%Y/%m/%d %H:%M:%S.%f%z'
            return datetime.strptime(time_stamp, date_format_str)
        except ValueError:
            date_format_str = '%Y/%m/%d %H:%M:%S.%f'
            return datetime.strptime(time_stamp, date_format_str)

    def get_time(self, sequence: pd.DataFrame, max_trace: int, mean_trace: int):
        list_seq = []
        for _, row in sequence.iterrows():
            list_temp = []
            seq = np.zeros(max_trace)
            for idx in range(len(row[0]) - 1):
                list_temp.append(self.to_sec(self.time_format(row[0][idx]) -
                                             self.time_format(row[0][0])))
                new_seq = np.append(seq, list_temp)
                cut = len(list_temp)
                new_seq = new_seq[cut:]
                list_seq.append(new_seq[-mean_trace:])
        return np.array(list_seq)

    def get_seq_view(self, sequence: pd.DataFrame, max_trace: int, mean_trace: int):
        list_seq = []
        for _, row in sequence.iterrows():
            list_temp = []
            seq = np.zeros(max_trace)
            for idx in range(len(row[0]) - 1):
                list_temp.append(row[0][idx])
                new_seq = np.append(seq, list_temp)
                cut = len(list_temp)
                new_seq = new_seq[cut:]
                list_seq.append(new_seq[-mean_trace:])
        return list_seq

    def get_sequence(self, sequence: pd.DataFrame, max_trace: int, mean_trace: int):
        list_seq = []
        list_label = []
        for _, row in sequence.iterrows():
            list_temp = []
            seq = np.zeros(max_trace)
            for idx in range(len(row[0]) - 1):
                list_temp.append(row[0][idx])
                new_seq = np.append(seq, list_temp)
                cut = len(list_temp)
                new_seq = new_seq[cut:]
                list_seq.append(new_seq[-mean_trace:])
                list_label.append(row[0][idx + 1])
        return list_seq, list_label

    def mapping(self, df_train: pd.DataFrame, df_test: pd.DataFrame, col: str):
        list_word = self.union(df_train[col].unique(), df_test[col].unique())
        return dict(zip(set(list_word), range(1, len(list_word) + 1)))

    def _save_numpy(self, name: str, array):
        np.save(self._data_dir / name, array)

    def mapping_cat(self, col: str, df_train: pd.DataFrame, df_test: pd.DataFrame,
                    max_trace: int, mean_trace: int, fold: int):
        target_train = df_train.groupby('case', sort=False).agg({col: lambda x: list(x)})
        target_test = df_test.groupby('case', sort=False).agg({col: lambda x: list(x)})

        if col == 'timestamp':
            scaler = MinMaxScaler()
            view_train = self.get_time(target_train, max_trace, mean_trace)
            view_test = self.get_time(target_test, max_trace, mean_trace)
            view_train = scaler.fit_transform(view_train)
            view_test = scaler.transform(view_test)
            self._save_numpy(f"{self._eventlog}_{col}_{fold}_train.npy", view_train)
            self._save_numpy(f"{self._eventlog}_{col}_{fold}_test.npy", view_test)
            self._list_num_cols.append(col)
            return

        if col == 'case':
            return

        if is_numeric_dtype(df_train[col]):
            scaler = MinMaxScaler()
            view_train = self.get_seq_view(target_train, max_trace, mean_trace)
            view_test = self.get_seq_view(target_test, max_trace, mean_trace)
            view_train = scaler.fit_transform(view_train)
            view_test = scaler.transform(view_test)
            self._save_numpy(f"{self._eventlog}_{col}_{fold}_train.npy", view_train)
            self._save_numpy(f"{self._eventlog}_{col}_{fold}_test.npy", view_test)
            self._list_num_cols.append(col)
            return

        mapping = self.mapping(df_train, df_test, col)
        df_train[col] = [mapping[item] for item in df_train[col]]
        df_test[col] = [mapping[item] for item in df_test[col]]
        view_train = df_train.groupby('case', sort=False).agg({col: lambda x: list(x)})
        view_test = df_test.groupby('case', sort=False).agg({col: lambda x: list(x)})

        if col == 'activity':
            view_train, label_train = self.get_sequence(view_train, max_trace, mean_trace)
            view_test, label_test = self.get_sequence(view_test, max_trace, mean_trace)
            self._save_numpy(f"{self._eventlog}_{col}_{fold}_train.npy", view_train)
            self._save_numpy(f"{self._eventlog}_{col}_{fold}_test.npy", view_test)
            self._save_numpy(f"{self._eventlog}_{col}_{fold}_info.npy", len(mapping))
            self._save_numpy(f"{self._eventlog}_y_{fold}_train.npy", label_train)
            self._save_numpy(f"{self._eventlog}_y_{fold}_test.npy", label_test)
            self._list_cat_cols.append(col)
            return

        view_train = self.get_seq_view(view_train, max_trace, mean_trace)
        view_test = self.get_seq_view(view_test, max_trace, mean_trace)
        self._save_numpy(f"{self._eventlog}_{col}_{fold}_train.npy", view_train)
        self._save_numpy(f"{self._eventlog}_{col}_{fold}_test.npy", view_test)
        self._save_numpy(f"{self._eventlog}_{col}_{fold}_info.npy", len(mapping))
        self._list_cat_cols.append(col)

    def read_view(self):
        for fold in range(3):
            self._list_cat_cols = []
            self._list_num_cols = []
            df_train = pd.read_csv(self._data_dir / f"{self._eventlog}_kfoldcv_{fold}_train.csv", sep=',')
            df_test = pd.read_csv(self._data_dir / f"{self._eventlog}_kfoldcv_{fold}_test.csv", sep=',')

            if self._eventlog in {'bpi12w_complete', 'bpi12_all_complete', 'bpi12_work_all'}:
                df_train['resource'] = 'Res' + df_train['resource'].astype(str)
                df_test['resource'] = 'Res' + df_test['resource'].astype(str)

            full_df = df_train.append(df_test)
            cont_trace = full_df['case'].value_counts(dropna=False)
            max_trace = max(cont_trace)
            mean_trace = int(round(np.mean(cont_trace)))
            for col in df_train.columns:
                self.mapping_cat(col, df_train, df_test, max_trace, mean_trace, fold)

        with open(self._data_dir / f"{self._eventlog}_seq_length.pickle", 'wb') as handle:
            pickle.dump(mean_trace, handle, protocol=pickle.HIGHEST_PROTOCOL)

        with open(self._data_dir / f"{self._eventlog}_cat_cols.pickle", 'wb') as handle:
            pickle.dump(self._list_cat_cols, handle, protocol=pickle.HIGHEST_PROTOCOL)

        with open(self._data_dir / f"{self._eventlog}_num_cols.pickle", 'wb') as handle:
            pickle.dump(self._list_num_cols, handle, protocol=pickle.HIGHEST_PROTOCOL)

        return {
            'categorical_columns': self._list_cat_cols,
            'numerical_columns': self._list_num_cols,
            'sequence_length': mean_trace,
        }


## ۴. کلاس اصلی مدل (`MiDA`)

این کلاس از فایل `MiDA.py` برداشته شده و برای استفاده در Colab به‌روزرسانی شده است.

In [ ]:

class MiDA:
    def __init__(self, eventlog: str, data_root: Path | str = "fold", models_dir: Path | str = "models"):
        self._eventlog = eventlog
        self._data_root = Path(data_root)
        self._data_dir = self._data_root / eventlog
        self._models_dir = Path(models_dir)
        self._models_dir.mkdir(parents=True, exist_ok=True)

        self._cat_view = []
        self._num_view = []
        self._seq_length = 0

        self.list_cat_view_train = []
        self.list_num_view_train = []
        self.y_train = []
        self.y_test = []
        self.n_classes = 0
        self.n_fold = 0
        self.best_score = np.inf
        self.best_model = None
        self.best_time = 0
        self.best_numparameters = 0
        self._best_temp_file = Path("best_temp.txt")

    @staticmethod
    def union(lst1, lst2):
        return lst1 + lst2

    def load_col(self):
        with open(self._data_dir / f"{self._eventlog}_num_cols.pickle", 'rb') as pickle_file:
            self._num_view = pickle.load(pickle_file)
        with open(self._data_dir / f"{self._eventlog}_cat_cols.pickle", 'rb') as pickle_file:
            self._cat_view = pickle.load(pickle_file)
        with open(self._data_dir / f"{self._eventlog}_seq_length.pickle", 'rb') as pickle_file:
            self._seq_length = pickle.load(pickle_file)

    def get_model(self, cfg):
        list_cat_view = []
        list_num_view = []
        list_cat_view_in = []
        list_num_view_in = []

        for c in self._cat_view:
            num_view = np.load(self._data_dir / f"{self._eventlog}_{c}_{0}_info.npy")
            size_view = num_view + 1 // 2
            input_cat = Input(shape=(self._seq_length,), dtype='int32', name=c)
            list_cat_view_in.append(input_cat)
            x = Embedding(output_dim=size_view, input_dim=num_view + 1, input_length=self._seq_length)(input_cat)
            list_cat_view.append(x)

        for n in self._num_view:
            input_num = Input(shape=(self._seq_length,), dtype='float32', name=n)
            x = Reshape((self._seq_length, 1))(input_num)
            list_num_view_in.append(input_num)
            list_num_view.append(x)

        layer_in = concatenate(self.union(list_cat_view, list_num_view))

        layer_l = LSTM(units=int(cfg["lstmsize1"]), kernel_initializer='glorot_uniform',
                       return_sequences=True, dropout=0.1, recurrent_dropout=0.1)(layer_in)
        layer_l = BatchNormalization()(layer_l)
        layer_l = LSTM(units=int(cfg["lstmsize2"]), kernel_initializer='glorot_uniform',
                       return_sequences=False, dropout=0.1, recurrent_dropout=0.1)(layer_l)
        layer_l = BatchNormalization()(layer_l)

        out = Dense(self.n_classes, activation='softmax')(layer_l)
        opt = Nadam(learning_rate=cfg['learning_rate_init'], beta_1=0.9, beta_2=0.999,
                    epsilon=1e-08, schedule_decay=0.004, clipvalue=3)
        model = Model(inputs=self.union(list_cat_view_in, list_num_view_in), outputs=out)
        model.compile(optimizer=opt, loss='categorical_crossentropy', metrics=['acc'])
        return model

    def fit_and_score(self, cfg):
        print(cfg)
        outfile_path = Path(f"{self._eventlog}_{self.n_fold}.txt")
        outfile_path.parent.mkdir(parents=True, exist_ok=True)
        with outfile_path.open('a') as outfile:
            start_time = perf_counter()
            model = self.get_model(cfg)
            early_stopping = EarlyStopping(monitor='val_loss', patience=20)
            lr_reducer = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=10, verbose=0,
                                           mode='auto', min_delta=0.0001, cooldown=0, min_lr=0)
            list_view = self.union(self.list_cat_view_train, self.list_num_view_train)
            history = model.fit(list_view, self.y_train, epochs=200, verbose=1,
                                validation_split=0.2, callbacks=[early_stopping, lr_reducer],
                                batch_size=cfg['batch_size'])
            scores = [history.history['val_loss'][epoch] for epoch in range(len(history.history['loss']))]
            score = min(scores)
            end_time = perf_counter()

            if self._best_temp_file.exists():
                current_best = float(self._best_temp_file.read_text())
            else:
                current_best = np.inf

            print("best_score->", current_best)
            print("score->", score)
            if current_best > score:
                self.best_score = score
                self.best_model = model
                self._best_temp_file.write_text(str(self.best_score))
                self.best_numparameters = model.count_params()
                self.best_time = end_time - start_time
                model.save(self._models_dir / f"{self._eventlog}model_smac_{self.n_fold}_layer.h5")

            outfile.write(f"{score};{len(history.history['loss'])};{model.count_params()};{end_time - start_time};"
                          f"{cfg['lstmsize1']};{cfg['lstmsize2']};{cfg['batch_size']};{cfg['learning_rate_init']}
")
        return score

    def smac_opt(self, max_evals: int = 20, max_iters: int = 200):
        self.load_col()
        for f in range(3):
            self.n_fold = f
            print('Starting model selection...')
            self.best_score = np.inf
            self.best_model = None
            self.best_time = 0
            self.best_numparameters = 0
            self._best_temp_file.write_text(str(np.inf))

            logging.basicConfig(level=logging.INFO)

            cs = ConfigurationSpace()
            lstmsize1 = CategoricalHyperparameter("lstmsize1", [50, 75, 100])
            lstmsize2 = CategoricalHyperparameter("lstmsize2", [50, 75, 100])
            batch_size = CategoricalHyperparameter("batch_size", [32, 64, 128, 256, 512, 1024])
            learning_rate_init = UniformFloatHyperparameter('learning_rate_init', 0.00001, 0.01,
                                                            default_value=0.001, log=True)
            cs.add_hyperparameters([lstmsize1, lstmsize2, batch_size, learning_rate_init])

            scenario = Scenario({
                "run_obj": "quality",
                "runcount-limit": max_evals,
                "cs": cs,
                "deterministic": "true",
                "abort_on_first_run_crash": "false",
                "output_dir": str(self._data_dir)
            })

            self.list_cat_view_train = [
                np.load(self._data_dir / f"{self._eventlog}_{col}_{f}_train.npy")
                for col in self._cat_view
            ]
            self.list_cat_view_test = [
                np.load(self._data_dir / f"{self._eventlog}_{col}_{f}_test.npy")
                for col in self._cat_view
            ]
            self.list_num_view_train = [
                np.load(self._data_dir / f"{self._eventlog}_{col}_{f}_train.npy", allow_pickle=True)
                for col in self._num_view
            ]
            self.list_num_view_test = [
                np.load(self._data_dir / f"{self._eventlog}_{col}_{f}_test.npy", allow_pickle=True)
                for col in self._num_view
            ]

            y_train = np.load(self._data_dir / f"{self._eventlog}_y_{f}_train.npy")
            y_test = np.load(self._data_dir / f"{self._eventlog}_y_{f}_test.npy")

            df_labels = np.unique(list(y_train) + list(y_test))
            label_encoder = preprocessing.LabelEncoder()
            integer_encoded = label_encoder.fit_transform(df_labels).reshape(-1, 1)

            onehot_encoder = preprocessing.OneHotEncoder(sparse=False)
            onehot_encoder.fit(integer_encoded)
            train_integer_encoded = label_encoder.transform(y_train).reshape(-1, 1)
            train_onehot_encoded = onehot_encoder.transform(train_integer_encoded)
            self.y_train = np.asarray(train_onehot_encoded)

            test_integer_encoded = label_encoder.transform(y_test).reshape(-1, 1)
            test_onehot_encoded = onehot_encoder.transform(test_integer_encoded)
            self.y_test = np.asarray(test_onehot_encoded)
            self.Y_test_int = np.asarray(test_integer_encoded)
            self.n_classes = len(df_labels)

            intensifier_kwargs = {'initial_budget': 20, 'max_budget': max_iters, 'eta': 3}
            print("Optimizing! Depending on your machine, this might take a few minutes.")
            smac = HB4AC(scenario=scenario,
                         rng=np.random.RandomState(42),
                         tae_runner=self.fit_and_score,
                         intensifier_kwargs=intensifier_kwargs)

            try:
                incumbent = smac.optimize()
            finally:
                incumbent = smac.solver.incumbent
            inc_value = smac.get_tae_runner().run(config=incumbent, instance='1', budget=max_iters, seed=0)[1]
            print(inc_value)
            print("Optimized Value: %.4f" % inc_value)


## ۵. توابع ارزیابی و گزارش

این قسمت نسخه‌ی Colab از `test.py` است.

In [ ]:

def union(lst1, lst2):
    return lst1 + lst2


def multiclass_roc_auc_score(y_test, y_pred, average):
    lb = LabelBinarizer()
    lb.fit(y_test)
    y_test = lb.transform(y_test)
    y_pred = lb.transform(y_pred)
    return roc_auc_score(y_test, y_pred, average=average)


def multiclass_pr_auc_score(y_test, y_pred, average):
    lb = LabelBinarizer()
    lb.fit(y_test)
    y_test = lb.transform(y_test)
    y_pred = lb.transform(y_pred)
    return average_precision_score(y_test, y_pred, average=average)


def evaluate_models(eventlog: str, data_root: Path | str = "fold", models_dir: Path | str = "models"):
    data_dir = Path(data_root) / eventlog
    models_dir = Path(models_dir)

    with open(data_dir / f"{eventlog}_num_cols.pickle", 'rb') as pickle_file:
        num_view = pickle.load(pickle_file)
    with open(data_dir / f"{eventlog}_cat_cols.pickle", 'rb') as pickle_file:
        cat_view = pickle.load(pickle_file)
    with open(data_dir / f"{eventlog}_seq_length.pickle", 'rb') as pickle_file:
        seq_length = pickle.load(pickle_file)

    metrics = []
    for f in range(3):
        model = load_model(models_dir / f"{eventlog}model_smac_{f}_layer.h5")
        list_cat_view_test = [
            np.load(data_dir / f"{eventlog}_{col}_{f}_test.npy")
            for col in cat_view
        ]
        list_num_view_test = [
            np.load(data_dir / f"{eventlog}_{col}_{f}_test.npy", allow_pickle=True)
            for col in num_view
        ]
        y_train = np.load(data_dir / f"{eventlog}_y_{f}_train.npy")
        y_test = np.load(data_dir / f"{eventlog}_y_{f}_test.npy")

        df_labels = np.unique(list(y_train) + list(y_test))
        label_encoder = preprocessing.LabelEncoder()
        integer_encoded = label_encoder.fit_transform(df_labels).reshape(-1, 1)
        onehot_encoder = preprocessing.OneHotEncoder(sparse=False)
        onehot_encoder.fit(integer_encoded)
        test_integer_encoded = label_encoder.transform(y_test).reshape(-1, 1)
        test_onehot_encoded = onehot_encoder.transform(test_integer_encoded)
        Y_test = np.asarray(test_onehot_encoded)
        Y_test_int = np.asarray(test_integer_encoded)

        list_view = union(list_cat_view_test, list_num_view_test)
        preds = model.predict(list_view)
        y_true = np.argmax(Y_test, axis=1)
        y_pred = np.argmax(preds, axis=1)

        precision, recall, fscore, _ = precision_recall_fscore_support(
            Y_test_int, y_pred, average='macro', pos_label=None
        )
        auc_score_macro = multiclass_roc_auc_score(Y_test_int, y_pred, average="macro")
        prauc_score_macro = multiclass_pr_auc_score(Y_test_int, y_pred, average="macro")

        report = classification_report(Y_test_int, y_pred, digits=3)
        print(f"Fold {f} report:
{report}")
        metrics.append({
            'fold': f,
            'classification_report': report,
            'precision': precision,
            'recall': recall,
            'f1_macro': fscore,
            'auc_macro': auc_score_macro,
            'prauc_macro': prauc_score_macro
        })

    return pd.DataFrame(metrics)


## ۶. اجرای کامل خط لوله

در این سلول ابتدا داده‌ها پیش‌پردازش می‌شوند و سپس بهینه‌سازی SMAC اجرا می‌گردد.

In [ ]:

# نام لاگ خود را اینجا قرار دهید
EVENT_LOG_NAME = "sample_eventlog"  # مثال: "bpi12w_complete"
DATA_ROOT = Path("fold")
MODELS_DIR = Path("models")

# آماده‌سازی پوشه‌ها
DATA_ROOT.mkdir(exist_ok=True)
MODELS_DIR.mkdir(exist_ok=True)

# ۱) اگر فایل‌های آماده‌سازی شده را ندارید، ابتدا داده‌ها را خوانده و پردازش کنید.
# در صورتی که فایل‌های numpy/pickle آماده دارید، می‌توانید از این مرحله صرف‌نظر کنید.
# reader = ReadLog(EVENT_LOG_NAME, DATA_ROOT)
# reader.read_view()

# ۲) سپس مدل را آموزش/بهینه کنید.
# mida = MiDA(EVENT_LOG_NAME, DATA_ROOT, MODELS_DIR)
# mida.smac_opt(max_evals=20, max_iters=200)

# ۳) در نهایت برای ارزیابی مدل‌های ذخیره‌شده:
# metrics_df = evaluate_models(EVENT_LOG_NAME, DATA_ROOT, MODELS_DIR)
# display(metrics_df)


> **نکته:** برای اجرای واقعی، سلول بالا را مطابق با داده‌های خود و با حذف علامت `#` از خطوط موردنیاز اجرا کنید.